# Text Feature Extraction for Vietnamese Real-Estate Listings

Turns the unstructured `name` / `description` fields of
[`tinixai/vietnam-real-estates`](https://huggingface.co/datasets/tinixai/vietnam-real-estates)
into model-ready features, and measures how much they add to a tabular price baseline.

**Three feature families**

1. **Domain keyword flags** (`kw_*`) — legal status (`sổ đỏ`, `sổ hồng`, `chính chủ`, `pháp lý rõ ràng`, `đang chờ sổ`), road accessibility (`ô tô đỗ cửa`, `mặt tiền`, `ngõ thông`, `xe hơi vào nhà`, `ngõ ba gác`) and condition/interior (`full nội thất`, `nội thất cơ bản`, `nhà mới`, `nhà cấp 4`). Matched diacritic-insensitively, with a negation guard so `không tranh chấp` does not read as a legal risk.
2. **Numeric entities** (`text_*`) — area, frontage × depth, floor count, bedrooms, bathrooms, road width and the advertised price, parsed out of free text and used to *impute* the heavily-sparse structured columns.
3. **Text representations** (`tfidf_*`, `phobert_*`) — TF-IDF + Truncated SVD by default, mean-pooled PhoBERT as an opt-in heavier path.

**Evaluation is out-of-time.** The parquet shards are chronologically ordered, so this trains on `shard_0000` (June 2025) and tests on `shard_0009` (March 2026) — a nine-month gap with no look-ahead.

Runtime: ~15 min on a Colab CPU runtime at the default sample sizes.

## 0. Setup

In [ ]:
%pip install -q lightgbm scikit-learn pandas pyarrow numpy

### 0.1 Bundled library

The four cells below inline the reachable subset of the project's
`real_estate/text/` package — text normalisation, the domain lexicon, entity
parsing, the TF-IDF / PhoBERT encoders, data loading and the uplift benchmark.
They are copied **verbatim** from those modules by
`notebooks/_build_text_features_notebook.py`, so this notebook runs on a bare
kernel with no project checkout and nothing to `pip install` from Git. Run them
once, top to bottom, then collapse the section.

In [ ]:
from __future__ import annotations

# == Bundled library: text normalisation, domain lexicon, keyword flags ==
# Copied verbatim from real_estate/text/ by notebooks/_build_text_features_notebook.py.
# Edit the source modules there, not here, then regenerate the notebook.

# ---- real_estate/text/normalize.py -------------------------------------------------

import re
import unicodedata


# Redaction placeholders emitted by the source crawler, e.g. `[phone_number]`.
_PLACEHOLDER_RE = re.compile(r"\[[a-z_]+\]")

# Symbol variants that are safe to normalise while *keeping* Vietnamese letters
# intact: superscripts, multiplication signs, dashes, quotes, ellipsis, NBSP.
# All entries are one character to one character so string length is preserved.
_SYMBOL_MAP = str.maketrans(
    {
        "\u00b2": "2",  # ²  (m² -> m2)
        "\u00b3": "3",  # ³
        "\u00d7": "x",  # ×  (4 × 12 -> 4 x 12)
        "\u2013": "-",  # –
        "\u2014": "-",  # —
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2026": " ",  # …
        "\u00a0": " ",  # non-breaking space
    }
)

# đ/Đ have no decomposed form, so they need an explicit map.  Applied only when
# folding — cleaning must leave them alone, since "đỏ" and "dỏ" are different
# words to any diacritic-sensitive model (PhoBERT included).
_D_MAP = str.maketrans({"\u0111": "d", "\u0110": "D"})

_MULTI_SPACE_RE = re.compile(r"[ \t\r\f\v]+")
_MULTI_NEWLINE_RE = re.compile(r"\n{3,}")


def _fold_char(ch: str) -> str:
    """Fold a single character to its ASCII base form (possibly empty)."""
    decomposed = unicodedata.normalize("NFD", ch.translate(_SYMBOL_MAP).translate(_D_MAP))
    kept = "".join(c for c in decomposed if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", kept).lower()


def fold_with_alignment(text: str) -> tuple[str, list[int]]:
    """Fold ``text`` and record which source character produced each output character.

    Folding is *not* length-preserving in general: combining marks are dropped,
    so a stray ``U+0301`` or an emoji variation selector (``U+FE0F``, category
    ``Mn``) makes the result shorter than its input, and real listing text does
    contain both.  Returning the per-character origin lets a span found in folded
    text be mapped back onto the original string exactly, which is what price
    redaction on the diacritic-preserving PhoBERT path needs.

    Returns ``(folded, origin)`` where ``origin[i]`` is the index in ``text`` of
    the character that produced ``folded[i]``.
    """
    pieces: list[str] = []
    origin: list[int] = []
    for index, ch in enumerate(text):
        folded = _fold_char(ch)
        pieces.append(folded)
        origin.extend([index] * len(folded))
    return "".join(pieces), origin


def fold_diacritics(text: str) -> str:
    """Lower-case and strip Vietnamese tone marks and diacritics.

    ``"Sổ đỏ chính chủ"`` and ``"so do chinh chu"`` both map to
    ``"so do chinh chu"``, which lets a single pattern match listings written
    with or without diacritics.  Use :func:`fold_with_alignment` when the folded
    offsets must be mapped back onto the original string.
    """
    return fold_with_alignment(text)[0]


def clean_listing_text(text: str | None) -> str:
    """Tidy a raw listing field without touching its Vietnamese letters.

    Diacritics — including ``đ`` — are preserved because they carry lexical
    meaning (``số`` vs ``sổ``, ``đỏ`` vs ``dỏ``); only whitespace, redaction
    placeholders and symbol variants are normalised.
    """
    if not text:
        return ""
    cleaned = _PLACEHOLDER_RE.sub(" ", text.translate(_SYMBOL_MAP))
    cleaned = _MULTI_NEWLINE_RE.sub("\n\n", cleaned)
    cleaned = _MULTI_SPACE_RE.sub(" ", cleaned)
    return cleaned.strip()


def make_searchable(text: str | None) -> str:
    """Clean and diacritic-fold ``text`` for regex matching."""
    return fold_diacritics(clean_listing_text(text))


def join_text_columns(values: object) -> str:
    """Join the text columns of one row into a single searchable string.

    Accepts ``None``, a plain string, or any iterable of them.  Columns are
    separated by a space so that n-gram features can span the title/description
    boundary, while regex matches stay anchored to real words.
    """
    if values is None:
        return ""
    if isinstance(values, str):
        return values
    return " ".join(v for v in values if isinstance(v, str) and v)

# ---- real_estate/text/lexicon.py ---------------------------------------------------

from dataclasses import dataclass



@dataclass(frozen=True)
class KeywordRule:
    """One matchable domain concept.

    Attributes
    ----------
    name:
        Feature suffix.  The extractor emits ``kw_<name>``.
    category:
        One of :data:`CATEGORIES`.
    label:
        Human-readable Vietnamese phrase(s) this rule stands for.
    pattern:
        Regex matched against folded text.  Use ``\\s+`` between syllables and
        ``o\\s?to`` style optionals for compounds written with or without a space.
    negation_guard:
        When true the match is discarded if a negation cue (``không``, ``chưa``,
        ``ko``, ``kg``...) immediately precedes it.  Set for terms where the
        negated form is common and means the *opposite* — e.g. ``không tranh
        chấp`` ("no dispute") must not flag a legal risk.
    """

    name: str
    category: str
    label: str
    pattern: str
    negation_guard: bool = False


CATEGORIES: tuple[str, ...] = ("legal", "access", "condition", "intent")

LEXICON: tuple[KeywordRule, ...] = (
    # ---------------------------------------------------------------- legal
    KeywordRule("so_do", "legal", "sổ đỏ", r"\bso\s+do\b"),
    KeywordRule("so_hong", "legal", "sổ hồng", r"\bso\s+hong\b"),
    KeywordRule(
        "so_do_hong",
        "legal",
        "sổ đỏ / sổ hồng (either certificate)",
        r"\bso\s+(do|hong)\b",
    ),
    KeywordRule("chinh_chu", "legal", "chính chủ", r"\bchinh\s+chu\b"),
    KeywordRule("phap_ly", "legal", "pháp lý", r"\bphap\s+ly\b"),
    KeywordRule(
        "phap_ly_ro_rang",
        "legal",
        "pháp lý rõ ràng / đầy đủ",
        r"\bphap\s+ly\s+(ro\s+rang|day\s+du|ro|chuan|minh\s+bach|hoan\s+chinh)\b",
    ),
    KeywordRule(
        "dang_cho_so",
        "legal",
        "đang chờ sổ",
        r"\b(dang\s+)?cho\s+so\b|\bso\s+(dang\s+)?(cho|ve|lam)\b|\bchua\s+co\s+so\b",
    ),
    KeywordRule("giay_to", "legal", "giấy tờ (hợp lệ)", r"\bgiay\s+to\b"),
    KeywordRule("cong_chung", "legal", "công chứng", r"\bcong\s+chung\b"),
    KeywordRule("hop_dong", "legal", "hợp đồng (mua bán)", r"\bhop\s+dong\b"),
    KeywordRule(
        "tranh_chap",
        "legal",
        "tranh chấp",
        r"\btranh\s+chap\b",
        negation_guard=True,
    ),
    # Raw mention is ambiguous: "không quy hoạch" advertises the *absence* of a
    # planning encumbrance.  The guard turns the bare mention into a risk signal
    # and a separate rule captures the positive claim.
    KeywordRule(
        "quy_hoach",
        "legal",
        "(dính) quy hoạch",
        r"\bquy\s+hoach\b",
        negation_guard=True,
    ),
    KeywordRule(
        "khong_quy_hoach",
        "legal",
        "không quy hoạch / không tranh chấp",
        r"\b(khong|ko|kg)\s+(dinh\s+)?(quy\s+hoach|tranh\s+chap)\b",
    ),
    # --------------------------------------------------------------- access
    KeywordRule("mat_tien", "access", "mặt tiền", r"\bmat\s+tien\b"),
    KeywordRule(
        "o_to_do_cua",
        "access",
        "ô tô đỗ cửa / đỗ cổng",
        r"\bo\s?to\s+(do|dau|ghe)\s+(cua|cong|nha|cua\s+nha)\b",
    ),
    KeywordRule(
        "xe_hoi_vao_nha",
        "access",
        "xe hơi vào nhà",
        r"\b(xe\s+hoi|o\s?to)\s+(vao|do|chay|den)\s+(nha|trong\s+nha|tan\s+nha|trong\s+trong)\b",
    ),
    KeywordRule("o_to", "access", "ô tô (bất kỳ ngữ cảnh nào)", r"\bo\s?to\b"),
    KeywordRule("xe_hoi", "access", "xe hơi", r"\bxe\s+hoi\b"),
    KeywordRule("ngo_thong", "access", "ngõ thông", r"\bngo\s+thong\b|\bhem\s+thong\b"),
    KeywordRule(
        "ngo_ba_gac",
        "access",
        "ngõ ba gác",
        r"\b(ngo|hem|ngo\s+hem)\s+(ba\s+gac|3\s+gac|bagac)\b|\bxe\s+ba\s+gac\b|\bba\s+gac\b",
    ),
    KeywordRule("hem", "access", "hẻm", r"\bhem\b"),
    KeywordRule("hem_xe_hoi", "access", "hẻm xe hơi", r"\bhem\s+(xe\s+hoi|o\s?to)\b"),
    KeywordRule("via_he", "access", "vỉa hè", r"\bvia\s+he\b|\ble\s+duong\b"),
    KeywordRule("phan_lo", "access", "khu phân lô", r"\bphan\s+lo\b"),
    KeywordRule("hai_mat_thoang", "access", "2 mặt thoáng", r"\b(2|hai)\s+mat\s+thoang\b"),
    KeywordRule("lo_goc", "access", "lô góc / căn góc", r"\b(lo|can|nha|dat)\s+goc\b"),
    KeywordRule("duong_rong", "access", "đường rộng / đường lớn", r"\bduong\s+(rong|lon|to)\b|\bduong\s+\d+\s*m\b"),
    # ------------------------------------------------------------ condition
    KeywordRule(
        "full_noi_that",
        "condition",
        "full nội thất / nội thất đầy đủ",
        r"\b(full|day\s+du|tron)\s+noi\s+that\b|\bnoi\s+that\s+(day\s+du|full|cao\s+cap|sang\s+trong|xach\s+vali)\b",
    ),
    KeywordRule("noi_that_co_ban", "condition", "nội thất cơ bản", r"\bnoi\s+that\s+co\s+ban\b"),
    KeywordRule("noi_that", "condition", "nội thất (bất kỳ)", r"\bnoi\s+that\b"),
    KeywordRule(
        "nha_moi",
        "condition",
        "nhà mới / xây mới",
        r"\bnha\s+moi\b|\bmoi\s+xay\b|\bxay\s+moi\b|\bmoi\s+coong\b|\bmoi\s+tinh\b",
    ),
    KeywordRule("nha_cap_4", "condition", "nhà cấp 4", r"\bnha\s+cap\s*4\b|\bcap\s*4\b"),
    KeywordRule("nha_cu", "condition", "nhà cũ / xuống cấp", r"\bnha\s+(cu|tho|nat)\b|\bxuong\s+cap\b"),
    KeywordRule("o_ngay", "condition", "vào ở ngay", r"\b(vao\s+)?o\s+ngay\b|\bdon\s+vao\s+o\b"),
    KeywordRule("thang_may", "condition", "thang máy", r"\bthang\s+may\b"),
    KeywordRule("san_thuong", "condition", "sân thượng", r"\bsan\s+thuong\b"),
    KeywordRule("tang_ham", "condition", "tầng hầm / bán hầm", r"\btang\s+ham\b|\bban\s+ham\b|\bham\s+(xe|o\s?to|gara)\b"),
    # --------------------------------------------------------------- intent
    # MEASURES A MENTION, NOT A LISTING TYPE. On shard_0000 this fires on 18.8%
    # of rows, but 96% of those carry a sale-scale price (median 10 tỷ VND): they
    # are sale listings pitching rental yield ("sẵn hợp đồng thuê", "tiện cho
    # thuê"), not rentals. Genuine rental listings quote a monthly rate (15-60M
    # VND) and are excluded by clean_frame's price floor instead. Use this as a
    # feature, never as a row filter.
    KeywordRule(
        "cho_thue",
        "intent",
        "nhắc đến cho thuê (không phân loại được tin thuê)",
        r"\bcho\s+thue\b|\bgia\s+thue\b|\bthue\s+nguyen\s+can\b|\bcan\s+ho\s+cho\s+thue\b",
    ),
    KeywordRule("kinh_doanh", "intent", "tiện kinh doanh", r"\bkinh\s+doanh\b|\bbuon\s+ban\b|\bcho\s+thue\s+kinh\s+doanh\b"),
    KeywordRule("ban_gap", "intent", "bán gấp / bán nhanh", r"\bban\s+(gap|nhanh|lo)\b|\bcan\s+ban\s+(gap|nhanh)\b|\bcan\s+tien\s+ban\s+gap\b"),
    KeywordRule("gia_re", "intent", "giá rẻ / giá tốt", r"\bgia\s+(re|tot|mem|nhe)\b"),
    KeywordRule("thuong_luong", "intent", "có thương lượng", r"\bthuong\s+luong\b|\bco\s+thuong\s+luong\b|\bgia\s+co\s+the\s+thuong\b"),
)

#: Aggregate flags derived from primitive rules.  Keys are emitted feature
#: suffixes; values are the primitive rule names they combine (OR semantics).
DERIVED_FLAGS: dict[str, tuple[str, ...]] = {
    "legal_certificate": ("so_do", "so_hong"),
    "legal_risk": ("tranh_chap", "quy_hoach", "dang_cho_so"),
    "legal_clean": ("phap_ly_ro_rang", "so_do_hong", "cong_chung"),
    "road_car_access": ("o_to_do_cua", "xe_hoi_vao_nha", "hem_xe_hoi"),
    "furnished": ("full_noi_that", "noi_that"),
}


def rules_by_category(category: str) -> tuple[KeywordRule, ...]:
    """Return the rules belonging to ``category``."""
    return tuple(r for r in LEXICON if r.category == category)

# ---- real_estate/text/keywords.py --------------------------------------------------

import re
from collections.abc import Iterable, Sequence

import pandas as pd



DEFAULT_TEXT_COLUMNS: tuple[str, ...] = ("name", "description")

_FEATURE_PREFIX = "kw_"

# Negation cues that must appear immediately before a guarded match to cancel it.
# Matched against the tail of the preceding window, so "không dính quy hoạch"
# suppresses the `quy_hoach` risk flag.
_NEGATION_TAIL_RE = re.compile(
    r"\b(khong|ko|kg|k0|chang|chua|khong\s+he|khong\s+bi|khong\s+dinh|mie[n]?)\b\s*(\w{1,6}\s*)?$"
)


class KeywordExtractor:
    """Extract boolean domain flags from listing text.

    Parameters
    ----------
    rules:
        Lexicon entries to apply.  Defaults to the full :data:`LEXICON`.
    derived_flags:
        OR-aggregates of primitive rule names, e.g. ``legal_certificate``.
    negation_window:
        Number of trailing folded characters inspected for a negation cue.
    text_columns:
        Columns concatenated per row before matching.
    """

    def __init__(
        self,
        rules: Sequence[KeywordRule] = LEXICON,
        derived_flags: dict[str, tuple[str, ...]] | None = None,
        negation_window: int = 28,
        text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    ) -> None:
        self.rules: tuple[KeywordRule, ...] = tuple(rules)
        self.derived_flags = dict(DERIVED_FLAGS if derived_flags is None else derived_flags)
        self.negation_window = int(negation_window)
        self.text_columns: tuple[str, ...] = tuple(text_columns)

        self._validate()
        self._compiled = {r.name: re.compile(r.pattern) for r in self.rules}

    # ------------------------------------------------------------------ setup
    def _validate(self) -> None:
        seen: set[str] = set()
        known = {r.name for r in self.rules}
        for rule in self.rules:
            if rule.name in seen:
                raise ValueError(f"duplicate lexicon rule name: {rule.name!r}")
            seen.add(rule.name)
            if rule.category not in CATEGORIES:
                raise ValueError(
                    f"rule {rule.name!r} has unknown category {rule.category!r}; "
                    f"expected one of {CATEGORIES}"
                )
            try:
                re.compile(rule.pattern)
            except re.error as exc:  # pragma: no cover - lexicon is static data
                raise ValueError(f"rule {rule.name!r} has an invalid pattern: {exc}") from exc
        for flag, sources in self.derived_flags.items():
            if flag in known:
                raise ValueError(f"derived flag {flag!r} collides with a primitive rule name")
            missing = [s for s in sources if s not in known]
            if missing:
                raise ValueError(f"derived flag {flag!r} references unknown rules: {missing}")

    @property
    def feature_names(self) -> list[str]:
        """Ordered output column names, primitive rules first."""
        return [f"{_FEATURE_PREFIX}{r.name}" for r in self.rules] + [
            f"{_FEATURE_PREFIX}{name}" for name in self.derived_flags
        ]

    # -------------------------------------------------------------- matching
    def _rule_hits(self, rule: KeywordRule, folded: str) -> bool:
        rx = self._compiled[rule.name]
        if not rule.negation_guard:
            return rx.search(folded) is not None
        if rx.search(folded) is None:
            return False
        for match in rx.finditer(folded):
            window = folded[max(0, match.start() - self.negation_window) : match.start()]
            if not _NEGATION_TAIL_RE.search(window):
                return True
        return False

    def extract_folded(self, folded_texts: Iterable[str]) -> pd.DataFrame:
        """Build the flag block from text that is already clean and folded."""
        rows: list[dict[str, bool]] = []
        for folded in folded_texts:
            values = {rule.name: self._rule_hits(rule, folded) for rule in self.rules}
            for flag, sources in self.derived_flags.items():
                values[flag] = any(values[s] for s in sources)
            rows.append({f"{_FEATURE_PREFIX}{k}": v for k, v in values.items()})
        flags = pd.DataFrame(rows, dtype=bool)
        if flags.empty:
            return pd.DataFrame({name: pd.Series(dtype=bool) for name in self.feature_names})
        return flags[self.feature_names]

    def extract(self, df: pd.DataFrame) -> pd.DataFrame:
        """Build the flag block for ``df`` using :attr:`text_columns`.

        The returned frame keeps ``df``'s index so it can be concatenated with
        tabular features via ``pd.concat(..., axis=1)``.
        """
        missing = [c for c in self.text_columns if c not in df.columns]
        if missing:
            raise KeyError(f"text column(s) not present in frame: {missing}")
        folded = (
            fold_diacritics(clean_listing_text(join_text_columns(row)))
            for row in df[list(self.text_columns)].itertuples(index=False, name=None)
        )
        return self.extract_folded(folded).set_axis(df.index)

    # -------------------------------------------------------------- reporting
    def prevalence(self, flags: pd.DataFrame) -> pd.DataFrame:
        """Per-flag hit counts and rates — used to validate the lexicon on real data."""
        labels = {r.name: r.label for r in self.rules}
        categories = {r.name: r.category for r in self.rules}
        n = len(flags)
        records = []
        for column in flags.columns:
            name = column[len(_FEATURE_PREFIX) :]
            hits = int(flags[column].sum())
            records.append(
                {
                    "feature": column,
                    "category": categories.get(name, "derived"),
                    "label": labels.get(name, " / ".join(self.derived_flags.get(name, ()))),
                    "hits": hits,
                    "rate": (hits / n) if n else float("nan"),
                }
            )
        return pd.DataFrame(records).sort_values("rate", ascending=False).reset_index(drop=True)

In [ ]:
from __future__ import annotations

# == Bundled library: numeric entity parsing and dense text representations ==
# Copied verbatim from real_estate/text/ by notebooks/_build_text_features_notebook.py.
# Edit the source modules there, not here, then regenerate the notebook.

# ---- real_estate/text/entities.py --------------------------------------------------

import re
from collections.abc import Iterable

import numpy as np
import pandas as pd



#: Entities that restate the regression target and must not be fed to a model
#: trained on that target.  ``price`` is written verbatim into most listing
#: descriptions ("giá: 7.45 tỷ"), so parsing it back out and offering it as a
#: feature is circular — it measures transcription, not valuation.
TARGET_LEAKING_ENTITIES: tuple[str, ...] = ("text_price_vnd",)

#: Replacement token for redacted price mentions.
PRICE_REDACTION = " <gia> "

#: Columns emitted by :func:`extract_entities`.
ENTITY_COLUMNS: tuple[str, ...] = (
    "text_price_vnd",
    "text_area_m2",
    "text_frontage_m",
    "text_depth_m",
    "text_floor_count",
    "text_lau_count",
    "text_bedroom_count",
    "text_bathroom_count",
    "text_road_width_m",
)

#: Structured columns that text entities may fill, mapped to their entity source.
TEXT_IMPUTABLE_COLUMNS: dict[str, str] = {
    "area": "text_area_m2",
    "frontage_width": "text_frontage_m",
    "house_depth": "text_depth_m",
    "floor_count": "text_floor_count",
    "bedroom_count": "text_bedroom_count",
    "bathroom_count": "text_bathroom_count",
    "road_width": "text_road_width_m",
}

# A number token using either separator style: 4 | 7.45 | 1,3 | 1.300.000
_NUM = r"\d+(?:[.,]\d+)+"
_INT = r"\d+"
_ANY_NUM = rf"(?:{_NUM}|{_INT})"

_AREA_RE = re.compile(
    rf"\b({_ANY_NUM})\s*(?:m2|mv|met\s+vuong|metro\s+vuong)\b", re.IGNORECASE
)
# "4m x 12m", "4 x 12", "4,5x16m"
_DIMS_RE = re.compile(
    rf"\b({_ANY_NUM})\s*m?\s*[x*]\s*({_ANY_NUM})\s*m?\b"
)
# "ngang 4m ... dai 12m"
_NGANG_DAI_RE = re.compile(rf"\bngang\s+({_ANY_NUM})\s*m?\b[^.]{{0,40}}?\bdai\s+({_ANY_NUM})\s*m?\b")
_TANG_RE = re.compile(rf"\b({_INT})\s*(?:tang|tam)\b")
_LAU_RE = re.compile(rf"\b({_INT})\s+lau\b")
_BED_RE = re.compile(rf"\b({_INT})\s*(?:pn|phong\s+ngu|bedroom)\b")
_BATH_RE = re.compile(rf"\b({_INT})\s*(?:wc|toilet|phong\s+tam|nvs|vs)\b")
# "duong 8m", "hem 3m", "ngo 2.5m", "mat duong 12m"
_ROAD_RE = re.compile(rf"\b(?:duong|hem|ngo|mat\s+duong|mt\s+duong|lo\s+gioi)\s+({_ANY_NUM})\s*(?:m|met)\b")

# Price with an explicit Vietnamese magnitude word.  "ty"/"ti" = 1e9, "tr" = 1e6.
_PRICE_UNITS: tuple[tuple[re.Pattern[str], float], ...] = (
    (re.compile(rf"\b({_ANY_NUM})\s*(?:ty|ti)\b"), 1e9),
    (re.compile(rf"\b({_ANY_NUM})\s*(?:trieu|tr)\b"), 1e6),
    (re.compile(rf"\b({_ANY_NUM})\s*(?:nghin|ngan|k)\b"), 1e3),
)
_GIA_RE = re.compile(r"\bgia\b")

# Sanity bounds: reject parses that cannot describe a real listing.
_MAX_AREA_M2 = 100_000.0
_MAX_DIM_M = 500.0
_MAX_FLOORS = 200.0
_MAX_ROOMS = 200.0
_MAX_ROAD_M = 200.0
_MAX_PRICE_VND = 1e15
_MIN_PRICE_VND = 1e5


def parse_vn_number(token: str) -> float | None:
    """Parse a Vietnamese-styled numeric token into a float.

    ``"7.45"`` -> 7.45, ``"1,3"`` -> 1.3, ``"1.300.000"`` -> 1300000.0,
    ``"1.300"`` -> 1300.0 (a 3-digit group is treated as thousands), ``"4"`` -> 4.0.
    """
    token = token.strip()
    if not token:
        return None
    parts = re.split(r"[.,]", token)
    if len(parts) == 1:
        return float(parts[0])
    if len(parts) == 2:
        whole, frac = parts
        # 1-2 digits after a single separator is a decimal; 3 digits is a
        # thousands group ("1.300" = one thousand three hundred).
        if 1 <= len(frac) <= 2:
            return float(f"{whole}.{frac}")
        return float(whole + frac)
    # Multiple separators can only be thousands grouping.
    return float("".join(parts))


def _first(pattern: re.Pattern[str], text: str, *, limit: float | None = None) -> float | None:
    match = pattern.search(text)
    if not match:
        return None
    value = parse_vn_number(match.group(1))
    if value is None:
        return None
    if limit is not None and not (0 < value <= limit):
        return None
    return value


def _extract_price(folded: str) -> float | None:
    """Return the advertised price in VND, preferring a value stated near "giá"."""
    best: tuple[float, float] | None = None  # (distance penalty, value)
    gia = _GIA_RE.search(folded)
    anchor = gia.start() if gia else None
    for pattern, multiplier in _PRICE_UNITS:
        for match in pattern.finditer(folded):
            amount = parse_vn_number(match.group(1))
            if amount is None:
                continue
            value = amount * multiplier
            if not (_MIN_PRICE_VND <= value <= _MAX_PRICE_VND):
                continue
            penalty = abs(match.start() - anchor) if anchor is not None else 1e6
            candidate = (penalty, value)
            if best is None or candidate[0] < best[0]:
                best = candidate
            if anchor is None:
                # Without a "giá" anchor the first stated amount wins.
                return value
    return best[1] if best else None


def _extract_floors(folded: str) -> float | None:
    """Return the level count from an explicit "N tầng" / "N tấm".

    ``N lầu`` is deliberately *not* converted here: southern listings use it
    both for "N levels above ground" and for "N levels total", so adding a
    ground floor would bake in an unfounded assumption.  ``text_lau_count`` is
    exposed as its own raw signal instead, and only the unambiguous
    ``tầng``/``tấm`` form imputes the structured ``floor_count`` column.
    """
    return _first(_TANG_RE, folded, limit=_MAX_FLOORS)


# ------------------------------------------------------------- price redaction
def find_price_spans(folded: str) -> list[tuple[int, int]]:
    """Locate advertised-price mentions in diacritic-folded text.

    Returns merged, left-to-right ``(start, end)`` spans covering each numeric
    amount and its magnitude word (``"7.45 ty"``, ``"900 trieu"``).
    """
    spans: list[tuple[int, int]] = []
    for pattern, _ in _PRICE_UNITS:
        spans.extend((m.start(), m.end()) for m in pattern.finditer(folded))
    if not spans:
        return []
    spans.sort()
    merged = [spans[0]]
    for start, end in spans[1:]:
        if start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged


def redact_spans(
    text: str, spans: Iterable[tuple[int, int]], replacement: str = PRICE_REDACTION
) -> str:
    """Replace each span in ``text`` with ``replacement``, right to left."""
    out = text
    for start, end in sorted(spans, reverse=True):
        out = out[:start] + replacement + out[end:]
    return out


def prepare_for_tfidf(text: str | None, redact_price: bool = True) -> str:
    """Clean, diacritic-fold and price-redact text for the TF-IDF view.

    Folding is applied because the TF-IDF vocabulary benefits from unifying
    spelling variants (``ô tô`` / ``ôtô``), and redaction removes the advertised
    price so bag-of-words features cannot smuggle the regression target in as
    digit n-grams.
    """
    folded = fold_diacritics(clean_listing_text(text))
    if redact_price:
        folded = redact_spans(folded, find_price_spans(folded))
    return folded


def _map_spans(spans: Iterable[tuple[int, int]], origin: list[int]) -> list[tuple[int, int]]:
    """Translate spans in folded text back into spans in the source string."""
    mapped = []
    for start, end in spans:
        if start < 0 or start >= end or end > len(origin):
            continue
        mapped.append((origin[start], origin[end - 1] + 1))
    return mapped


def prepare_for_embedding(text: str | None, redact_price: bool = True) -> str:
    """Clean and price-redact text for a diacritic-sensitive encoder (PhoBERT).

    PhoBERT was trained on properly diacritised Vietnamese, so diacritics are
    *kept* here; price spans are located on the folded form and mapped back onto
    the cleaned string per character, because folding is not length-preserving
    (stray combining marks and emoji variation selectors are dropped).
    """
    cleaned = clean_listing_text(text)
    if not redact_price:
        return cleaned
    folded, origin = fold_with_alignment(cleaned)
    return redact_spans(cleaned, _map_spans(find_price_spans(folded), origin))


def extract_entities_folded(folded_texts: Iterable[str]) -> pd.DataFrame:
    """Extract numeric entities from already clean, diacritic-folded text."""
    rows: list[dict[str, float]] = []
    for folded in folded_texts:
        area = _first(_AREA_RE, folded, limit=_MAX_AREA_M2)
        dims = _DIMS_RE.search(folded) or _NGANG_DAI_RE.search(folded)
        frontage = depth = None
        if dims:
            f = parse_vn_number(dims.group(1))
            d = parse_vn_number(dims.group(2))
            if f is not None and 0 < f <= _MAX_DIM_M:
                frontage = f
            if d is not None and 0 < d <= _MAX_DIM_M:
                depth = d
        rows.append(
            {
                "text_price_vnd": _extract_price(folded),
                "text_area_m2": area,
                "text_frontage_m": frontage,
                "text_depth_m": depth,
                "text_floor_count": _extract_floors(folded),
                "text_lau_count": _first(_LAU_RE, folded, limit=_MAX_FLOORS),
                "text_bedroom_count": _first(_BED_RE, folded, limit=_MAX_ROOMS),
                "text_bathroom_count": _first(_BATH_RE, folded, limit=_MAX_ROOMS),
                "text_road_width_m": _first(_ROAD_RE, folded, limit=_MAX_ROAD_M),
            }
        )
    frame = pd.DataFrame(rows, columns=list(ENTITY_COLUMNS), dtype="float64")
    return frame


def extract_entities(df: pd.DataFrame, text_columns: tuple[str, ...] = ("name", "description")) -> pd.DataFrame:
    """Extract numeric entities for ``df``, preserving its index."""
    missing = [c for c in text_columns if c not in df.columns]
    if missing:
        raise KeyError(f"text column(s) not present in frame: {missing}")
    folded = (
        fold_diacritics(clean_listing_text(join_text_columns(row)))
        for row in df[list(text_columns)].itertuples(index=False, name=None)
    )
    return extract_entities_folded(folded).set_axis(df.index)


def impute_from_text(tabular: pd.DataFrame, entities: pd.DataFrame) -> pd.DataFrame:
    """Fill null structured columns from text-derived entities.

    Returns a copy containing the imputed structured columns plus, for every
    imputed column, a ``<column>__from_text`` boolean recording that the value
    came from free text rather than the source record.  Existing non-null values
    are never overwritten, so the transformation is monotone in information.
    """
    if not tabular.index.equals(entities.index):
        entities = entities.reindex(tabular.index)
    out = tabular.copy()
    for column, source in TEXT_IMPUTABLE_COLUMNS.items():
        if column not in out.columns or source not in entities.columns:
            continue
        original = pd.to_numeric(out[column], errors="coerce").astype("float64")
        fill = entities[source].astype("float64")
        filled = np.where(original.notna() & (original > 0), original, fill)
        out[column] = pd.Series(filled, index=out.index, dtype="float64")
        out[f"{column}__from_text"] = pd.Series(
            (original.isna() | (original <= 0)) & fill.notna(),
            index=out.index,
            dtype=bool,
        )
    return out

# ---- real_estate/text/representation.py --------------------------------------------

from collections.abc import Iterable, Sequence

import numpy as np
import pandas as pd
from scipy.sparse import issparse
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer



TFIDF_MODES: tuple[str, ...] = ("word", "char", "both")


def _as_text_list(texts: Iterable[str | None]) -> list[str]:
    return [clean_listing_text(t) for t in texts]


class TfidfTextEncoder:
    """TF-IDF + Truncated SVD encoder for Vietnamese listing text.

    Vietnamese is syllable-delimited, so word n-grams are needed to capture
    multi-syllable domain phrases (``sổ đỏ``, ``mặt tiền``); character n-grams add
    robustness to abbreviation and teencode variants (``ô tô`` / ``ôtô`` / ``oto``).

    Parameters
    ----------
    mode:
        ``"word"``, ``"char"`` or ``"both"`` (independent SVD per view, concatenated).
    n_components:
        SVD dimensions kept per view.
    max_features, ngram_range, min_df, sublinear_tf:
        Passed through to :class:`~sklearn.feature_extraction.text.TfidfVectorizer`.
        ``ngram_range`` applies to the active analyzer's own units.
    """

    def __init__(
        self,
        mode: str = "word",
        n_components: int = 96,
        max_features: int = 60_000,
        ngram_range: tuple[int, int] = (1, 2),
        min_df: int = 5,
        sublinear_tf: bool = True,
        seed: int = 0,
    ) -> None:
        if mode not in TFIDF_MODES:
            raise ValueError(f"mode must be one of {TFIDF_MODES}, got {mode!r}")
        if n_components < 1:
            raise ValueError("n_components must be >= 1")
        self.mode = mode
        self.n_components = int(n_components)
        self.max_features = int(max_features)
        self.ngram_range = tuple(ngram_range)
        self.min_df = int(min_df)
        self.sublinear_tf = bool(sublinear_tf)
        self.seed = int(seed)

        self._views: dict[str, TfidfVectorizer] = {}
        self._svd: dict[str, TruncatedSVD] = {}
        self._fitted = False

    # ------------------------------------------------------------------ setup
    def _view_specs(self) -> dict[str, dict[str, object]]:
        specs: dict[str, dict[str, object]] = {}
        if self.mode in ("word", "both"):
            specs["w"] = {
                "analyzer": "word",
                "token_pattern": r"(?u)\b\w+\b",
                "ngram_range": self.ngram_range,
            }
        if self.mode in ("char", "both"):
            specs["c"] = {"analyzer": "char_wb", "ngram_range": (3, 5)}
        return specs

    @property
    def feature_prefixes(self) -> list[str]:
        """Column-name prefixes produced by this encoder, in output order."""
        if not self._fitted:
            return []
        return [f"tfidf_{key}" for key in self._views]

    @property
    def vocabulary_sizes(self) -> dict[str, int]:
        """Fitted vocabulary size per view (empty before :meth:`fit`)."""
        return {key: len(vec.vocabulary_) for key, vec in self._views.items()}

    @property
    def explained_variance(self) -> dict[str, float]:
        """Total SVD explained-variance ratio per view."""
        return {
            key: float(np.sum(svd.explained_variance_ratio_))
            for key, svd in self._svd.items()
        }

    # ---------------------------------------------------------------- fitting
    def fit(self, texts: Iterable[str | None]) -> TfidfTextEncoder:
        """Fit every TF-IDF view and its SVD on ``texts``."""
        rows = _as_text_list(texts)
        if not rows:
            raise ValueError("cannot fit TfidfTextEncoder on an empty corpus")
        self._views, self._svd = {}, {}
        for key, extra in self._view_specs().items():
            vectorizer = TfidfVectorizer(
                max_features=self.max_features,
                min_df=self.min_df,
                sublinear_tf=self.sublinear_tf,
                lowercase=True,
                strip_accents="unicode",
                **extra,  # type: ignore[arg-type]
            )
            matrix = vectorizer.fit_transform(rows)
            n_components = min(self.n_components, matrix.shape[1] - 1, matrix.shape[0] - 1)
            if n_components < 1:
                raise ValueError(
                    f"corpus too small for SVD on view {key!r}: "
                    f"{matrix.shape[0]} documents x {matrix.shape[1]} terms"
                )
            svd = TruncatedSVD(n_components=n_components, random_state=self.seed)
            svd.fit(matrix)
            self._views[key] = vectorizer
            self._svd[key] = svd
        self._fitted = True
        return self

    def transform(self, texts: Iterable[str | None], frame: bool = True) -> pd.DataFrame | np.ndarray:
        """Project ``texts`` into the fitted dense representation."""
        if not self._fitted:
            raise RuntimeError("TfidfTextEncoder.transform called before fit")
        rows = _as_text_list(texts)
        blocks = []
        for key, vectorizer in self._views.items():
            matrix = vectorizer.transform(rows)
            blocks.append(self._svd[key].transform(matrix))
        if not blocks:  # pragma: no cover - guarded by fit()
            raise RuntimeError("no TF-IDF views were fitted")
        dense = np.hstack(blocks).astype(np.float32)
        if not frame:
            return dense
        columns = [
            f"tfidf_{key}_{i}"
            for key in self._views
            for i in range(self._svd[key].n_components)
        ]
        return pd.DataFrame(dense, columns=columns)

    def fit_transform(self, texts: Iterable[str | None], frame: bool = True) -> pd.DataFrame | np.ndarray:
        """Fit on ``texts`` and return their representation."""
        rows = _as_text_list(texts)
        self.fit(rows)
        return self.transform(rows, frame=frame)

    # ------------------------------------------------------------------ utils
    def top_terms(self, view: str = "w", component: int = 0, k: int = 12) -> list[tuple[str, float]]:
        """Highest-loading terms of an SVD component — for qualitative inspection."""
        if view not in self._views:
            raise KeyError(f"unknown TF-IDF view {view!r}; fitted views: {list(self._views)}")
        vectorizer, svd = self._views[view], self._svd[view]
        if not (0 <= component < svd.n_components):
            raise IndexError(f"component {component} out of range for {svd.n_components} components")
        names = np.array(vectorizer.get_feature_names_out())
        weights = svd.components_[component]
        order = np.argsort(-np.abs(weights))[:k]
        return [(str(names[i]), float(weights[i])) for i in order]


class PhoBERTEmbedder:
    """Mean-pooled PhoBERT embeddings for listing text.

    ``torch`` and ``transformers`` are imported on first use, so the rest of the
    package works without them installed.

    Parameters
    ----------
    model_name:
        Hugging Face id of a PhoBERT-style encoder.
    max_length, batch_size:
        Token budget and inference batch size (CPU-bound: keep ``batch_size`` modest).
    pooling:
        ``"mean"`` over unmasked tokens, or ``"cls"`` for the ``<s>`` vector.
    n_components:
        When set, embeddings are reduced with Truncated SVD fitted on the first
        :meth:`fit` corpus, which keeps the downstream tabular model tractable.
    """

    def __init__(
        self,
        model_name: str = "vinai/phobert-base",
        max_length: int = 128,
        batch_size: int = 16,
        pooling: str = "mean",
        n_components: int | None = None,
        seed: int = 0,
        device: str = "cpu",
    ) -> None:
        if pooling not in ("mean", "cls"):
            raise ValueError(f"pooling must be 'mean' or 'cls', got {pooling!r}")
        self.model_name = model_name
        self.max_length = int(max_length)
        self.batch_size = int(batch_size)
        self.pooling = pooling
        self.n_components = None if n_components is None else int(n_components)
        self.seed = int(seed)
        self.device = device

        self._tokenizer = None
        self._model = None
        self._svd: TruncatedSVD | None = None
        self.hidden_size_: int | None = None

    # ------------------------------------------------------------------ setup
    def _ensure_model(self) -> None:
        if self._model is not None:
            return
        try:
            import torch  # noqa: F401  (imported for the availability check)
            from transformers import AutoModel, AutoTokenizer
        except ImportError as exc:  # pragma: no cover - depends on environment
            raise ImportError(
                "PhoBERTEmbedder requires 'torch' and 'transformers'. Install them with "
                "`uv pip install torch transformers` (CPU wheel: "
                "`--index-url https://download.pytorch.org/whl/cpu`), or use "
                "TfidfTextEncoder instead."
            ) from exc
        from transformers import AutoModel, AutoTokenizer

        self._tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self._model = AutoModel.from_pretrained(self.model_name)
        self._model.eval()
        self._model.to(self.device)
        self.hidden_size_ = int(self._model.config.hidden_size)

    # ---------------------------------------------------------------- encode
    def encode(self, texts: Iterable[str | None]) -> np.ndarray:
        """Return raw pooled embeddings, shape ``(len(texts), hidden_size)``."""
        self._ensure_model()
        import torch

        rows = _as_text_list(texts)
        out = np.zeros((len(rows), self.hidden_size_), dtype=np.float32)
        for start in range(0, len(rows), self.batch_size):
            chunk = rows[start : start + self.batch_size]
            encoded = self._tokenizer(
                chunk,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            ).to(self.device)
            with torch.no_grad():
                hidden = self._model(**encoded).last_hidden_state
            mask = encoded["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            if self.pooling == "mean":
                pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
            else:
                pooled = hidden[:, 0]
            out[start : start + len(chunk)] = pooled.cpu().numpy()
        return out

    # ---------------------------------------------------------------- fitting
    def fit(self, texts: Iterable[str | None]) -> PhoBERTEmbedder:
        """Fit the optional SVD reduction on ``texts``."""
        embeddings = self.encode(texts)
        if self.n_components is not None:
            n_components = min(self.n_components, embeddings.shape[1], embeddings.shape[0] - 1)
            if n_components < 1:
                raise ValueError("corpus too small for PhoBERT SVD reduction")
            self._svd = TruncatedSVD(n_components=n_components, random_state=self.seed)
            self._svd.fit(embeddings)
        return self

    def transform(self, texts: Iterable[str | None], frame: bool = True) -> pd.DataFrame | np.ndarray:
        """Embed ``texts`` (and reduce them when ``n_components`` is set)."""
        embeddings = self.encode(texts)
        if self._svd is not None:
            embeddings = self._svd.transform(embeddings).astype(np.float32)
        if not frame:
            return embeddings
        return pd.DataFrame(embeddings, columns=[f"phobert_{i}" for i in range(embeddings.shape[1])])

    def fit_transform(self, texts: Iterable[str | None], frame: bool = True) -> pd.DataFrame | np.ndarray:
        """Fit on ``texts`` and return their embedding representation."""
        rows = _as_text_list(texts)
        embeddings = self.encode(rows)
        if self.n_components is not None:
            n_components = min(self.n_components, embeddings.shape[1], len(rows) - 1)
            if n_components < 1:
                raise ValueError("corpus too small for PhoBERT SVD reduction")
            self._svd = TruncatedSVD(n_components=n_components, random_state=self.seed)
            embeddings = self._svd.fit_transform(embeddings).astype(np.float32)
        if not frame:
            return embeddings
        return pd.DataFrame(embeddings, columns=[f"phobert_{i}" for i in range(embeddings.shape[1])])


def hstack_blocks(blocks: Sequence[pd.DataFrame | np.ndarray]) -> pd.DataFrame:
    """Column-bind feature blocks, accepting sparse matrices, arrays or frames."""
    frames: list[pd.DataFrame] = []
    for block in blocks:
        if block is None:
            continue
        if isinstance(block, pd.DataFrame):
            frames.append(block)
        elif issparse(block):
            frames.append(pd.DataFrame(block.toarray()))
        else:
            frames.append(pd.DataFrame(np.asarray(block)))
    if not frames:
        raise ValueError("no feature blocks to concatenate")
    return pd.concat(frames, axis=1)


def ensure_dense(matrix: object) -> np.ndarray:
    """Return ``matrix`` as a dense float array (accepts sparse input)."""
    if issparse(matrix):
        return np.asarray(matrix.toarray())
    return np.asarray(matrix)

In [ ]:
from __future__ import annotations

# == Bundled library: dataset loading / cleaning and the feature pipeline ==
# Copied verbatim from real_estate/text/ by notebooks/_build_text_features_notebook.py.
# Edit the source modules there, not here, then regenerate the notebook.

# ---- real_estate/text/data.py ------------------------------------------------------

import unicodedata
import urllib.request
from collections.abc import Iterable, Sequence
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


DATASET_ID = "tinixai/vietnam-real-estates"
SHARD_URL_TEMPLATE = (
    "https://huggingface.co/datasets/" + DATASET_ID + "/resolve/main/{name}.parquet"
)

TEXT_COLUMNS: tuple[str, ...] = ("name", "description")
TARGET_COLUMN = "price"
TIME_COLUMN = "published_at"

NUMERIC_COLUMNS: tuple[str, ...] = (
    "area",
    "floor_count",
    "frontage_width",
    "house_depth",
    "road_width",
    "bedroom_count",
    "bathroom_count",
)

CATEGORICAL_COLUMNS: tuple[str, ...] = (
    "property_type_name",
    "province_name",
    "district_name",
    "ward_name",
    "street_name",
    "project_name",
    "house_direction",
    "balcony_direction",
)

DEFAULT_COLUMNS: tuple[str, ...] = (
    TEXT_COLUMNS + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS + (TARGET_COLUMN, TIME_COLUMN)
)


def resolve_shard(source: str | Path, cache_dir: str | Path | None = None) -> Path:
    """Return a local path for ``source``, downloading it when it names a shard.

    ``source`` may be an existing local path, a bare shard name such as
    ``"shard_0000"``, or a full URL.  Downloads land in ``cache_dir``.
    """
    text = str(source)
    if text.startswith(("http://", "https://")):
        url = text
        name = Path(text.split("?")[0]).stem
    else:
        candidate = Path(text)
        if candidate.exists():
            return candidate
        url = SHARD_URL_TEMPLATE.format(name=candidate.name)
        name = candidate.stem

    cache = Path(cache_dir) if cache_dir is not None else Path.cwd() / "data"
    cache.mkdir(parents=True, exist_ok=True)
    destination = cache / f"{name}.parquet"
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    tmp = destination.with_suffix(".part")
    with urllib.request.urlopen(url) as response, open(tmp, "wb") as handle:  # noqa: S310
        while chunk := response.read(1 << 22):
            handle.write(chunk)
    tmp.rename(destination)
    return destination


def parse_price(series: pd.Series) -> pd.Series:
    """Convert the string-typed ``price`` column to float VND.

    Parsing is deliberately strict: anything that is not a plain number becomes
    ``NaN`` rather than being coerced.  Stripping currency words and separators
    would silently turn ``"7.45 tỷ"`` into 7.45 **VND**, a billion-fold error, so
    ambiguous values are dropped for the caller to notice.  On the published
    shards every non-null value is a plain integer string.
    """
    if pd.api.types.is_numeric_dtype(series):
        return series.astype("float64")
    cleaned = series.astype("string").str.strip()
    return pd.to_numeric(cleaned, errors="coerce").astype("float64")


def normalize_unicode(frame: pd.DataFrame) -> pd.DataFrame:
    """NFC-normalise every string column, in place, and return the frame.

    About 0.15% of ``street_name`` and ``project_name`` values are stored in
    NFD-decomposed form (a base letter plus separate combining marks).  They
    render identically but compare unequal, so the same street silently becomes
    two categories in any frequency or target encoding.  The geographic columns
    that matter most (``province_name``, ``district_name``, ``ward_name``) are
    already clean NFC; this is cheap insurance for the rest.
    """
    for column in frame.columns:
        if not pd.api.types.is_string_dtype(frame[column]):
            continue
        frame[column] = frame[column].map(
            lambda value: unicodedata.normalize("NFC", value) if isinstance(value, str) else value
        )
    return frame


def load_shard_sample(
    source: str | Path,
    n_rows: int | None = None,
    seed: int = 0,
    columns: Sequence[str] | None = None,
    cache_dir: str | Path | None = None,
) -> pd.DataFrame:
    """Load a uniform random sample of one parquet shard.

    Only ``columns`` are read off disk.  With ``n_rows`` set, a seeded random row
    subset is taken before conversion to pandas so peak memory stays bounded.
    """
    path = resolve_shard(source, cache_dir=cache_dir)
    wanted = list(columns) if columns is not None else list(DEFAULT_COLUMNS)
    table = pq.read_table(path, columns=wanted)
    total = table.num_rows
    if n_rows is not None and n_rows < total:
        rng = np.random.default_rng(seed)
        indices = np.sort(rng.choice(total, size=int(n_rows), replace=False))
        table = table.take(indices)
    frame = normalize_unicode(table.to_pandas())
    # Column projection may exclude these; only parse what was actually read.
    if TARGET_COLUMN in frame.columns:
        frame[TARGET_COLUMN] = parse_price(frame[TARGET_COLUMN])
    if TIME_COLUMN in frame.columns:
        frame[TIME_COLUMN] = pd.to_datetime(frame[TIME_COLUMN], errors="coerce", format="ISO8601")
    return frame.reset_index(drop=True)


def clean_frame(
    df: pd.DataFrame,
    *,
    min_price: float = 1e8,
    max_price: float = 2e11,
    min_area: float = 15.0,
    max_area: float = 3_000.0,
    min_unit_price: float = 3e6,
    max_unit_price: float = 8e8,
    drop_mask: pd.Series | None = None,
) -> pd.DataFrame:
    """Drop records that cannot support a sale-price regression.

    Defaults match **Rule 1 ("Recommended Baseline")** in
    ``reports/vietnam_real_estates_eda.md`` so this module and the baseline
    notebook model the same population: price in [100M, 200B] VND, area in
    [15, 3000] m², and unit price in [3M, 800M] VND/m². On the full corpus that
    rule retains ~89.6% of rows and leaves log-price skew at +0.19.

    The unit-price band is what catches price/area mismatches that neither bound
    catches alone — a 20 tỷ listing recorded as 3 m², or a 500 m² flat priced
    like a parking space.

    The ``min_price`` floor is also what excludes genuine rental listings, which
    quote a *monthly* rate (15-60M VND on ``shard_0000``), plus token deposits.

    Do **not** filter rentals with the ``kw_cho_thue`` text flag. It measures a
    *mention*, and 96% of the rows it fires on carry a sale-scale price (median
    10 tỷ VND) — they are sale listings pitching rental yield ("sẵn hợp đồng
    thuê", "tiện cho thuê"). Dropping on it discards ~19% of valid training rows
    and does so non-randomly, since yield-pitched properties form a coherent
    higher-priced segment. ``drop_mask`` stays available for a caller that has a
    trustworthy exclusion signal.

    The original index is **preserved** so feature blocks computed on ``df`` stay
    row-aligned with the returned subset; call ``reset_index`` explicitly if a
    clean 0..n-1 index is wanted.
    """
    if TARGET_COLUMN not in df.columns:
        raise KeyError(f"frame is missing the target column {TARGET_COLUMN!r}")
    price = pd.to_numeric(df[TARGET_COLUMN], errors="coerce")
    keep = price.notna() & (price >= min_price) & (price <= max_price)
    if "area" in df.columns:
        area = pd.to_numeric(df["area"], errors="coerce")
        keep &= area.notna() & (area >= min_area) & (area <= max_area)
        unit_price = price / area.where(area > 0)
        keep &= unit_price.notna() & (unit_price >= min_unit_price) & (unit_price <= max_unit_price)
    if drop_mask is not None:
        if not drop_mask.index.equals(df.index):
            raise ValueError("drop_mask index does not align with the frame index")
        keep &= ~drop_mask.astype(bool)
    return df.loc[keep]


def time_span(df: pd.DataFrame) -> tuple[str, str]:
    """Return the inclusive ``(min, max)`` ISO range of ``published_at``."""
    stamps = df[TIME_COLUMN].dropna()
    if stamps.empty:
        return ("n/a", "n/a")
    return (str(stamps.min()), str(stamps.max()))


def out_of_time_pair(
    train_source: str | Path,
    test_source: str | Path,
    n_train: int,
    n_test: int,
    seed: int = 0,
    columns: Iterable[str] | None = None,
    cache_dir: str | Path | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load an early shard for training and a later shard for testing.

    Rows are sampled *before* cleaning so the split has no look-ahead from the
    test period into the training period.
    """
    wanted = list(columns) if columns is not None else None
    train = load_shard_sample(
        train_source, n_rows=n_train, seed=seed, columns=wanted, cache_dir=cache_dir
    )
    test = load_shard_sample(
        test_source, n_rows=n_test, seed=seed + 1, columns=wanted, cache_dir=cache_dir
    )
    return train, test

# ---- real_estate/text/pipeline.py --------------------------------------------------

from collections.abc import Sequence
from dataclasses import dataclass, field

import pandas as pd



#: Named feature groups, used for ablation in the benchmark.
FEATURE_GROUPS: tuple[str, ...] = ("keywords", "entities", "tfidf", "phobert")


@dataclass
class TextFeatureConfig:
    """Knobs for :class:`TextFeaturePipeline`."""

    text_columns: tuple[str, ...] = DEFAULT_TEXT_COLUMNS
    use_keywords: bool = True
    use_entities: bool = True
    use_tfidf: bool = True
    use_phobert: bool = False
    tfidf_mode: str = "word"
    tfidf_components: int = 96
    tfidf_max_features: int = 60_000
    tfidf_ngram_range: tuple[int, int] = (1, 2)
    tfidf_min_df: int = 5
    fold_for_tfidf: bool = True
    redact_price: bool = True
    phobert: dict = field(default_factory=dict)
    seed: int = 0

    def groups(self) -> list[str]:
        """Enabled feature groups, in canonical order."""
        return [
            name
            for name, enabled in (
                ("keywords", self.use_keywords),
                ("entities", self.use_entities),
                ("tfidf", self.use_tfidf),
                ("phobert", self.use_phobert),
            )
            if enabled
        ]


class TextFeaturePipeline:
    """Fit text-feature state on a training frame and apply it anywhere.

    Examples
    --------
    >>> pipeline = TextFeaturePipeline(TextFeatureConfig(tfidf_components=8))
    >>> _ = pipeline.fit(train_df)
    >>> test_features = pipeline.transform(test_df)
    """

    def __init__(self, config: TextFeatureConfig | None = None) -> None:
        self.config = config or TextFeatureConfig()
        if not self.config.groups():
            raise ValueError("at least one text feature group must be enabled")
        self.keywords = (
            KeywordExtractor(text_columns=self.config.text_columns)
            if self.config.use_keywords
            else None
        )
        self.tfidf: TfidfTextEncoder | None = None
        self.phobert: PhoBERTEmbedder | None = None
        self._fitted = False

    # ------------------------------------------------------------------ setup
    def _joined_texts(self, df: pd.DataFrame) -> list[str]:
        missing = [c for c in self.config.text_columns if c not in df.columns]
        if missing:
            raise KeyError(f"text column(s) not present in frame: {missing}")
        joined = (
            df[list(self.config.text_columns)].astype("string").fillna("").agg(" ".join, axis=1)
        )
        return list(joined)

    def _tfidf_texts(self, df: pd.DataFrame) -> list[str]:
        """Text for the TF-IDF view: folded by default, price mentions redacted."""
        prepare = prepare_for_tfidf if self.config.fold_for_tfidf else prepare_for_embedding
        return [prepare(t, redact_price=self.config.redact_price) for t in self._joined_texts(df)]

    def _embedding_texts(self, df: pd.DataFrame) -> list[str]:
        """Text for PhoBERT: diacritics preserved, price mentions redacted."""
        return [
            prepare_for_embedding(t, redact_price=self.config.redact_price)
            for t in self._joined_texts(df)
        ]

    # ---------------------------------------------------------------- fitting
    def _build_encoders(self) -> None:
        """Instantiate the fitted encoders without fitting them."""
        if self.config.use_tfidf:
            self.tfidf = TfidfTextEncoder(
                mode=self.config.tfidf_mode,
                n_components=self.config.tfidf_components,
                max_features=self.config.tfidf_max_features,
                ngram_range=self.config.tfidf_ngram_range,
                min_df=self.config.tfidf_min_df,
                seed=self.config.seed,
            )
        if self.config.use_phobert:
            self.phobert = PhoBERTEmbedder(seed=self.config.seed, **self.config.phobert)

    def fit(self, df: pd.DataFrame) -> TextFeaturePipeline:
        """Fit TF-IDF / PhoBERT state on ``df``."""
        self._build_encoders()
        if self.tfidf is not None:
            self.tfidf.fit(self._tfidf_texts(df))
        if self.phobert is not None:
            self.phobert.fit(self._embedding_texts(df))
        self._fitted = True
        return self

    # -------------------------------------------------------------- transform
    def _assemble(
        self,
        df: pd.DataFrame,
        tfidf_block: pd.DataFrame | None = None,
        phobert_block: pd.DataFrame | None = None,
    ) -> pd.DataFrame:
        """Column-bind the stateless blocks with any precomputed representations."""
        blocks: list[pd.DataFrame] = []
        if self.keywords is not None:
            blocks.append(self.keywords.extract(df))

        if self.config.use_entities:
            entities = extract_entities(df, text_columns=self.config.text_columns)
            blocks.append(entities)
            blocks.append(self._provenance_flags(df, entities))

        if tfidf_block is not None:
            blocks.append(tfidf_block)
        if phobert_block is not None:
            blocks.append(phobert_block)

        out = pd.concat(blocks, axis=1)
        if len(out) != len(df):
            raise AssertionError(
                f"text feature block has {len(out)} rows for a {len(df)}-row frame; "
                "a sub-block is not index-aligned"
            )
        out.index = df.index
        return out

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return all enabled text-derived features for ``df``, index-aligned.

        Output column groups:

        ``kw_*``
            boolean domain flags.
        ``text_*``
            numeric entities parsed from the listing text.
        ``<col>__from_text``
            boolean provenance, true where a structured column was null and the
            text supplied a value (see :func:`real_estate.text.entities.impute_from_text`).
        ``tfidf_*`` / ``phobert_*``
            dense representation components.
        """
        if (self.config.use_tfidf or self.config.use_phobert) and not self._fitted:
            raise RuntimeError("TextFeaturePipeline.transform called before fit")

        # Representation encoders are index-agnostic and return a fresh
        # RangeIndex; re-attach df's labels so the axis=1 concat in _assemble
        # aligns row-for-row instead of taking an index union.
        tfidf_block = (
            self.tfidf.transform(self._tfidf_texts(df)).set_axis(df.index)
            if self.tfidf is not None
            else None
        )
        phobert_block = (
            self.phobert.transform(self._embedding_texts(df)).set_axis(df.index)
            if self.phobert is not None
            else None
        )
        return self._assemble(df, tfidf_block, phobert_block)

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fit on ``df`` and return its features without encoding its text twice.

        PhoBERT encoding dominates this pipeline's cost, so the fitted frame's
        representation is taken from the encoders' own ``fit_transform`` rather
        than recomputed by a following ``transform`` pass.
        """
        self._build_encoders()
        tfidf_block = (
            self.tfidf.fit_transform(self._tfidf_texts(df)).set_axis(df.index)
            if self.tfidf is not None
            else None
        )
        phobert_block = (
            self.phobert.fit_transform(self._embedding_texts(df)).set_axis(df.index)
            if self.phobert is not None
            else None
        )
        self._fitted = True
        return self._assemble(df, tfidf_block, phobert_block)

    # ------------------------------------------------------------------ utils
    def _provenance_flags(self, df: pd.DataFrame, entities: pd.DataFrame) -> pd.DataFrame:
        """Mark structured columns whose text-derived replacement would be used."""
        flags = {}
        for column, source in TEXT_IMPUTABLE_COLUMNS.items():
            if column not in df.columns or source not in entities.columns:
                continue
            original = pd.to_numeric(df[column], errors="coerce")
            flags[f"{column}__from_text"] = (original.isna() | (original <= 0)) & entities[source].notna()
        return pd.DataFrame(flags, index=df.index, dtype=bool)

    def group_columns(self, features: pd.DataFrame) -> dict[str, list[str]]:
        """Partition an emitted feature frame into its ablation groups."""
        columns = list(features.columns)
        groups: dict[str, list[str]] = {name: [] for name in FEATURE_GROUPS}
        for column in columns:
            if column.startswith("kw_"):
                groups["keywords"].append(column)
            elif column.startswith("tfidf_"):
                groups["tfidf"].append(column)
            elif column.startswith("phobert_"):
                groups["phobert"].append(column)
            elif column.startswith("text_") or column.endswith("__from_text"):
                groups["entities"].append(column)
        return {name: cols for name, cols in groups.items() if cols}

    @staticmethod
    def entity_columns() -> Sequence[str]:
        """Entity columns parsed from text, in canonical order."""
        return ENTITY_COLUMNS

In [ ]:
from __future__ import annotations

# == Bundled library: marginal-uplift benchmark (reference tabular baseline + LightGBM) ==
# Copied verbatim from real_estate/text/ by notebooks/_build_text_features_notebook.py.
# Edit the source modules there, not here, then regenerate the notebook.

# ---- real_estate/text/benchmark.py -------------------------------------------------

import argparse
import json
import time
from collections.abc import Sequence
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd



VND_PER_TY = 1e9  # one "tỷ" = 1e9 VND


# --------------------------------------------------------------------- metrics
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Price-prediction metrics in natural units.

    ``y_true`` / ``y_pred`` are VND prices.  RMSLE is the RMSE of ``log1p`` values,
    the primary objective; MAE and MedAE are additionally reported in tỷ VND
    because that is the unit Vietnamese listings quote in.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mask = np.isfinite(y_true) & np.isfinite(y_pred) & (y_true > 0) & (y_pred > 0)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if y_true.size == 0:
        metrics = {k: float("nan") for k in ("rmsle", "mape_pct", "mae_ty", "medae_ty", "r2")}
        metrics["n"] = 0
        return metrics

    log_true, log_pred = np.log1p(y_true), np.log1p(y_pred)
    abs_error = np.abs(y_pred - y_true)
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    return {
        "rmsle": float(np.sqrt(np.mean((log_true - log_pred) ** 2))),
        "mape_pct": float(np.mean(abs_error / y_true) * 100.0),
        "mae_ty": float(np.mean(abs_error) / VND_PER_TY),
        "medae_ty": float(np.median(abs_error) / VND_PER_TY),
        "r2": 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan"),
        "n": int(y_true.size),
    }


# ------------------------------------------------------------ tabular encoder
@dataclass
class TabularEncoder:
    """Reference tabular feature block: numerics, ratios and geographic encodings.

    Fitted state is the category frequency map and the smoothed per-category mean
    of ``log1p(price)``.  Unknown categories at transform time fall back to the
    training global mean, which keeps the encoding leak-free and total.
    """

    geo_target_columns: tuple[str, ...] = ("province_name", "district_name", "ward_name")
    freq_columns: tuple[str, ...] = (
        "property_type_name",
        "province_name",
        "district_name",
        "ward_name",
        "street_name",
        "project_name",
        "house_direction",
        "balcony_direction",
    )
    smoothing: float = 50.0
    freq_prefix: str = "freq_"
    target_prefix: str = "tgt_"
    _freq: dict = field(default_factory=dict, init=False, repr=False)
    _target: dict = field(default_factory=dict, init=False, repr=False)
    _global_mean: float = field(default=float("nan"), init=False, repr=False)

    @staticmethod
    def _numeric_block(df: pd.DataFrame) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        for column in NUMERIC_COLUMNS:
            out[column] = (
                pd.to_numeric(df[column], errors="coerce").astype("float64")
                if column in df.columns
                else np.nan
            )
        area, frontage = out.get("area"), out.get("frontage_width")
        depth, road = out.get("house_depth"), out.get("road_width")
        if frontage is not None and depth is not None:
            out["ratio_frontage_to_depth"] = frontage / depth.replace(0, np.nan)
        if road is not None and frontage is not None:
            out["ratio_road_to_frontage"] = road / frontage.replace(0, np.nan)
        if area is not None and frontage is not None:
            out["implied_depth_from_area"] = area / frontage.replace(0, np.nan)
        if TIME_COLUMN in df.columns:
            stamps = pd.to_datetime(df[TIME_COLUMN], errors="coerce")
            out["published_month"] = stamps.dt.year * 12 + stamps.dt.month
            out["published_day_of_week"] = stamps.dt.dayofweek
        return out

    def fit(self, df: pd.DataFrame) -> TabularEncoder:
        """Learn frequencies and smoothed target means from training rows."""
        log_price = np.log1p(pd.to_numeric(df[TARGET_COLUMN], errors="coerce").astype("float64"))
        self._global_mean = float(np.nanmean(log_price))
        for column in self.freq_columns:
            if column not in df.columns:
                continue
            values = df[column].astype("string").fillna("__missing__")
            counts = values.value_counts()
            self._freq[column] = counts
            if column in self.geo_target_columns:
                grouped = pd.DataFrame({"v": values, "y": log_price}).groupby("v")["y"]
                means, sizes = grouped.mean(), grouped.size()
                self._target[column] = (
                    (means * sizes + self._global_mean * self.smoothing) / (sizes + self.smoothing)
                )
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Encode ``df`` using fitted state only."""
        if not self._freq and not np.isfinite(self._global_mean):
            raise RuntimeError("TabularEncoder.transform called before fit")
        out = self._numeric_block(df)
        for column in self.geo_target_columns:
            if column in df.columns and column in self._target:
                values = df[column].astype("string").fillna("__missing__")
                out[f"{self.target_prefix}{column}"] = (
                    values.map(self._target[column]).astype("float64").fillna(self._global_mean)
                )
        for column, counts in self._freq.items():
            if column not in df.columns:
                continue
            values = df[column].astype("string").fillna("__missing__")
            encoded = values.map(counts).astype("float64")
            out[f"{self.freq_prefix}{column}"] = np.log1p(encoded.fillna(0.0))
        return out

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fit on ``df`` and return its encoding."""
        return self.fit(df).transform(df)


def build_reference_tabular(
    train: pd.DataFrame, test: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Convenience wrapper returning fitted train/test reference tabular blocks."""
    encoder = TabularEncoder().fit(train)
    return encoder.transform(train), encoder.transform(test)


# ------------------------------------------------------- entity sanity report
def entity_agreement_report(df: pd.DataFrame, entities: pd.DataFrame) -> pd.DataFrame:
    """How often does a text-parsed entity match the structured column?

    Reported per entity: coverage (text produced a value), overlap (both present),
    and agreement among overlapping rows.  ``price`` uses a 5% relative tolerance
    because listings round ("7.45 tỷ" against a recorded 7,450,000,000 VND);
    counts and dimensions must match exactly.
    """
    pairs = list(TEXT_IMPUTABLE_COLUMNS.items()) + [("price", "text_price_vnd")]
    rows = []
    for structured, parsed in pairs:
        if structured not in df.columns or parsed not in entities.columns:
            continue
        left = pd.to_numeric(df[structured], errors="coerce")
        right = entities[parsed]
        both = left.notna() & (left > 0) & right.notna()
        n_both = int(both.sum())
        if structured == "price":
            agree = (np.abs(left[both] - right[both]) / left[both].clip(lower=1)) <= 0.05
        else:
            agree = np.isclose(left[both], right[both], rtol=0.02, atol=0.51)
        rows.append(
            {
                "structured_column": structured,
                "text_entity": parsed,
                "text_coverage": float(right.notna().mean()),
                "structured_missing_rate": float((left.isna() | (left <= 0)).mean()),
                "overlap_rows": n_both,
                "agreement_rate": float(agree.mean()) if n_both else float("nan"),
                "recovered_when_missing": int(
                    ((left.isna() | (left <= 0)) & right.notna()).sum()
                ),
            }
        )
    return pd.DataFrame(rows)


# ------------------------------------------------------------------- modelling
def _time_ordered_valid(train: pd.DataFrame, valid_fraction: float) -> tuple[np.ndarray, np.ndarray]:
    """Split training rows by ``published_at`` so early stopping cannot see the end."""
    stamps = pd.to_datetime(train[TIME_COLUMN], errors="coerce")
    if stamps.isna().all():
        n_valid = max(1, int(len(train) * valid_fraction))
        return np.arange(len(train) - n_valid), np.arange(len(train) - n_valid, len(train))
    order = np.argsort(stamps.to_numpy(dtype="datetime64[ns]"), kind="stable")
    cut = int(len(order) * (1.0 - valid_fraction))
    return order[:cut], order[cut:]


def _fit_lgbm(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_valid: np.ndarray,
    y_valid: np.ndarray,
    x_test: np.ndarray,
    *,
    seed: int,
    n_estimators: int,
    learning_rate: float,
    num_leaves: int,
) -> np.ndarray:
    """Fit LightGBM on ``log1p(price)`` and return VND predictions for the test block."""
    import lightgbm as lgb

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        min_child_samples=40,
        subsample=0.9,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(
        x_train,
        y_train,
        eval_set=[(x_valid, y_valid)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
    )
    return np.expm1(model.predict(x_test))


def _feature_importance_top(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_valid: np.ndarray,
    y_valid: np.ndarray,
    names: Sequence[str],
    *,
    seed: int,
    n_estimators: int,
    learning_rate: float,
    num_leaves: int,
    top_k: int = 25,
) -> list[tuple[str, float]]:
    """Top-``k`` gain-importance features for the richest arm (diagnostic only)."""
    import lightgbm as lgb

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        min_child_samples=40,
        subsample=0.9,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
        importance_type="gain",
    )
    model.fit(
        x_train,
        y_train,
        eval_set=[(x_valid, y_valid)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
    )
    gains = model.booster_.feature_importance(importance_type="gain")
    order = np.argsort(-gains)[:top_k]
    total = float(gains.sum()) or 1.0
    return [(str(names[i]), float(gains[i] / total)) for i in order]


# ------------------------------------------------------------------ the ladder
@dataclass
class BenchmarkConfig:
    """Everything the ablation ladder needs to run."""

    n_train: int = 100_000
    n_test: int = 40_000
    seed: int = 0
    n_seeds: int = 2
    valid_fraction: float = 0.15
    n_estimators: int = 1_200
    learning_rate: float = 0.06
    num_leaves: int = 63
    exclude_target_leaking_entities: bool = True
    redact_price_in_text: bool = True
    tfidf_mode: str = "word"
    tfidf_components: int = 96
    tfidf_max_features: int = 60_000
    tfidf_min_df: int = 5
    use_phobert: bool = False
    phobert: dict = field(default_factory=dict)
    # Mirrors clean_frame's defaults, which follow the EDA's recommended Rule 1.
    min_price: float = 1e8
    max_price: float = 2e11


def run_uplift_benchmark(
    train_source: str | Path,
    test_source: str | Path,
    config: BenchmarkConfig | None = None,
    cache_dir: str | Path | None = None,
    tabular_train: pd.DataFrame | None = None,
    tabular_test: pd.DataFrame | None = None,
) -> dict:
    """Run the ablation ladder and return a JSON-serialisable report.

    Arms, each a superset of the previous:

    ``tabular``      reference tabular block only
    ``+keywords``    adds the ``kw_*`` domain flags
    ``+entities``    adds parsed numeric entities, provenance flags and imputation
    ``+tfidf``       adds TF-IDF/SVD components
    ``+phobert``     adds PhoBERT embeddings (only when ``use_phobert``)
    """
    config = config or BenchmarkConfig()
    started = time.time()

    keyword_extractor = KeywordExtractor()
    raw_train = load_shard_sample(
        train_source, n_rows=config.n_train, seed=config.seed, cache_dir=cache_dir
    )
    raw_test = load_shard_sample(
        test_source, n_rows=config.n_test, seed=config.seed + 1, cache_dir=cache_dir
    )
    # Flags are computed on the raw rows so prevalence reporting below reflects
    # the full corpus rather than the cleaned subset.
    raw_train_flags = keyword_extractor.extract(raw_train)
    raw_test_flags = keyword_extractor.extract(raw_test)

    # Cleaning is by price/area bounds only. The min_price floor is what excludes
    # genuine rental listings (monthly rates of 15-60M VND); the kw_cho_thue text
    # flag must NOT be used for this, because 96% of the rows it fires on are sale
    # listings pitching rental yield at a sale-scale price. See clean_frame.
    train = clean_frame(raw_train, min_price=config.min_price, max_price=config.max_price)
    test = clean_frame(raw_test, min_price=config.min_price, max_price=config.max_price)
    # clean_frame preserves labels, so this selects exactly the surviving rows.
    train_flags = raw_train_flags.loc[train.index]
    test_flags = raw_test_flags.loc[test.index]

    text_config = TextFeatureConfig(
        use_keywords=False,  # flags already computed above, before cleaning
        use_entities=True,
        use_tfidf=True,
        use_phobert=config.use_phobert,
        tfidf_mode=config.tfidf_mode,
        tfidf_components=config.tfidf_components,
        tfidf_max_features=config.tfidf_max_features,
        tfidf_min_df=config.tfidf_min_df,
        redact_price=config.redact_price_in_text,
        phobert=config.phobert,
        seed=config.seed,
    )
    pipeline = TextFeaturePipeline(text_config)
    train_text = pipeline.fit_transform(train)  # encodes the train text exactly once
    test_text = pipeline.transform(test)

    entity_cols = [c for c in ENTITY_COLUMNS if c in train_text.columns]
    # `text_price_vnd` restates the regression target — the asking price is written
    # into the listing itself — so it is measured for extraction quality but never
    # handed to the model.
    model_entity_cols = (
        [c for c in entity_cols if c not in TARGET_LEAKING_ENTITIES]
        if config.exclude_target_leaking_entities
        else entity_cols
    )
    provenance_cols = [c for c in train_text.columns if c.endswith("__from_text")]
    tfidf_cols = [c for c in train_text.columns if c.startswith("tfidf_")]
    phobert_cols = [c for c in train_text.columns if c.startswith("phobert_")]

    tab_encoder = TabularEncoder().fit(train)
    tab_train = tab_encoder.transform(train)
    tab_test = tab_encoder.transform(test)

    def with_text_imputation(frame: pd.DataFrame, text: pd.DataFrame) -> pd.DataFrame:
        """Substitute text-recovered values into the sparse structured columns.

        Imputation happens *before* tabular encoding so derived ratios
        (frontage/depth, road/frontage) are computed from the recovered numbers
        rather than from the original nulls.
        """
        numeric_present = [c for c in NUMERIC_COLUMNS if c in frame.columns]
        if not numeric_present or not model_entity_cols:
            return frame
        out = frame.copy()
        filled = impute_from_text(frame[numeric_present], text[model_entity_cols])
        for column in numeric_present:
            out[column] = filled[column]
        return out

    tab_train_entities = tab_encoder.transform(with_text_imputation(train, train_text))
    tab_test_entities = tab_encoder.transform(with_text_imputation(test, test_text))
    entity_train = train_text[model_entity_cols + provenance_cols]
    entity_test = test_text[model_entity_cols + provenance_cols]

    arms: dict[str, tuple[pd.DataFrame, pd.DataFrame]] = {
        "tabular": (tab_train, tab_test),
        "+keywords": (
            pd.concat([tab_train, train_flags], axis=1),
            pd.concat([tab_test, test_flags], axis=1),
        ),
        "+entities": (
            pd.concat([tab_train_entities, train_flags, entity_train], axis=1),
            pd.concat([tab_test_entities, test_flags, entity_test], axis=1),
        ),
    }
    if tfidf_cols:
        arms["+tfidf"] = (
            pd.concat([arms["+entities"][0], train_text[tfidf_cols]], axis=1),
            pd.concat([arms["+entities"][1], test_text[tfidf_cols]], axis=1),
        )
    if phobert_cols:
        base = arms.get("+tfidf", arms["+entities"])
        arms["+phobert"] = (
            pd.concat([base[0], train_text[phobert_cols]], axis=1),
            pd.concat([base[1], test_text[phobert_cols]], axis=1),
        )

    y_train_all = np.log1p(train[TARGET_COLUMN].to_numpy(dtype=np.float64))
    y_test = test[TARGET_COLUMN].to_numpy(dtype=np.float64)
    fit_idx, valid_idx = _time_ordered_valid(train, config.valid_fraction)

    results: dict[str, dict] = {}
    richest: str = list(arms)[-1]
    for arm_name, (x_tr, x_te) in arms.items():
        x_train_np = x_tr.to_numpy(dtype=np.float32, na_value=np.nan)
        x_test_np = x_te.to_numpy(dtype=np.float32, na_value=np.nan)
        per_seed = []
        for offset in range(config.n_seeds):
            seed = config.seed + offset
            predictions = _fit_lgbm(
                x_train_np[fit_idx],
                y_train_all[fit_idx],
                x_train_np[valid_idx],
                y_train_all[valid_idx],
                x_test_np,
                seed=seed,
                n_estimators=config.n_estimators,
                learning_rate=config.learning_rate,
                num_leaves=config.num_leaves,
            )
            per_seed.append(regression_metrics(y_test, predictions))
        metrics = {
            key: float(np.mean([run[key] for run in per_seed]))
            for key in per_seed[0]
            if key != "n"
        }
        spread = {
            f"{key}_std": float(np.std([run[key] for run in per_seed]))
            for key in ("rmsle", "mape_pct")
        }
        importance = []
        if arm_name == richest:
            importance = _feature_importance_top(
                x_train_np[fit_idx],
                y_train_all[fit_idx],
                x_train_np[valid_idx],
                y_train_all[valid_idx],
                list(x_tr.columns),
                seed=config.seed,
                n_estimators=config.n_estimators,
                learning_rate=config.learning_rate,
                num_leaves=config.num_leaves,
            )
        results[arm_name] = {
            "n_features": int(x_tr.shape[1]),
            "metrics": metrics,
            "seed_spread": spread,
            "per_seed": per_seed,
            "top_features": importance,
        }

    baseline = results["tabular"]["metrics"]
    deltas = {}
    for arm_name, payload in results.items():
        if arm_name == "tabular":
            continue
        metrics = payload["metrics"]
        deltas[arm_name] = {
            "rmsle_delta": metrics["rmsle"] - baseline["rmsle"],
            "rmsle_delta_pct": 100.0 * (metrics["rmsle"] - baseline["rmsle"]) / baseline["rmsle"],
            "mape_delta_pct_points": metrics["mape_pct"] - baseline["mape_pct"],
            "mae_ty_delta": metrics["mae_ty"] - baseline["mae_ty"],
            "r2_delta": metrics["r2"] - baseline["r2"],
        }

    agreement = entity_agreement_report(train, train_text[entity_cols])
    # Lexicon health is measured on the *raw* rows: cleaning removes price/area
    # outliers, and a prevalence table computed after that describes the surviving
    # subset rather than the corpus the lexicon has to cover.
    prevalence = keyword_extractor.prevalence(raw_train_flags)

    return {
        "config": asdict(config),
        "dataset": {
            "train_shard": str(train_source),
            "test_shard": str(test_source),
            "train_rows_raw": int(len(raw_train)),
            "train_rows_clean": int(len(train)),
            "test_rows_raw": int(len(raw_test)),
            "test_rows_clean": int(len(test)),
            "train_time_span": list(time_span(train)),
            "test_time_span": list(time_span(test)),
            "tfidf_vocabulary": pipeline.tfidf.vocabulary_sizes if pipeline.tfidf else {},
            "tfidf_explained_variance": pipeline.tfidf.explained_variance if pipeline.tfidf else {},
        },
        "arms": results,
        "deltas_vs_tabular": deltas,
        "entity_agreement": agreement.to_dict(orient="records"),
        "keyword_prevalence_basis": (
            f"raw training rows before cleaning (n={len(raw_train_flags):,}); cleaning drops "
            "price/area outliers, so prevalence is reported on the full corpus"
        ),
        "keyword_prevalence": prevalence.to_dict(orient="records"),
        "runtime_seconds": round(time.time() - started, 1),
    }


def format_report(report: dict) -> str:
    """Render a report as a compact markdown summary."""
    lines = ["## Text-feature uplift benchmark", ""]
    meta = report["dataset"]
    lines.append(
        f"- Train: {meta['train_rows_clean']:,} clean rows of {meta['train_rows_raw']:,} "
        f"({meta['train_time_span'][0][:10]} → {meta['train_time_span'][1][:10]})"
    )
    lines.append(
        f"- Test (out-of-time): {meta['test_rows_clean']:,} clean rows of {meta['test_rows_raw']:,} "
        f"({meta['test_time_span'][0][:10]} → {meta['test_time_span'][1][:10]})"
    )
    lines.append(f"- Runtime: {report['runtime_seconds']}s")
    lines += ["", "| arm | feats | RMSLE | MAPE % | MAE (tỷ) | MedAE (tỷ) | R² |",
              "|---|---|---|---|---|---|---|"]
    for arm_name, payload in report["arms"].items():
        m = payload["metrics"]
        lines.append(
            f"| {arm_name} | {payload['n_features']} | {m['rmsle']:.4f} | {m['mape_pct']:.2f} "
            f"| {m['mae_ty']:.3f} | {m['medae_ty']:.3f} | {m['r2']:.4f} |"
        )
    lines += ["", "### Marginal gain vs tabular-only", "",
              "| arm | ΔRMSLE | ΔRMSLE % | ΔMAPE (pts) | ΔMAE (tỷ) | ΔR² |", "|---|---|---|---|---|---|"]
    for arm_name, d in report["deltas_vs_tabular"].items():
        lines.append(
            f"| {arm_name} | {d['rmsle_delta']:+.4f} | {d['rmsle_delta_pct']:+.2f}% "
            f"| {d['mape_delta_pct_points']:+.2f} | {d['mae_ty_delta']:+.3f} | {d['r2_delta']:+.4f} |"
        )
    lines += ["", "### Entity extraction agreement (train rows)", "",
              "| structured | text entity | text coverage | col missing | overlap | agreement | recovered |",
              "|---|---|---|---|---|---|---|"]
    for row in report["entity_agreement"]:
        agree = row["agreement_rate"]
        agree_s = f"{agree:.3f}" if agree == agree else "n/a"
        lines.append(
            f"| {row['structured_column']} | {row['text_entity']} | {row['text_coverage']:.3f} "
            f"| {row['structured_missing_rate']:.3f} | {row['overlap_rows']:,} | {agree_s} "
            f"| {row['recovered_when_missing']:,} |"
        )
    top = report["arms"][list(report["arms"])[-1]].get("top_features") or []
    if top:
        lines += ["", "### Top-15 gain-importance features (richest arm)", ""]
        for name, share in top[:15]:
            lines.append(f"- `{name}` — {share * 100:.2f}% of total gain")
    return "\n".join(lines)


def main(argv: Sequence[str] | None = None) -> int:
    """CLI: run the uplift benchmark and write JSON + markdown reports."""
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--train-shard", default="shard_0000")
    parser.add_argument("--test-shard", default="shard_0009")
    parser.add_argument("--cache-dir", default="data")
    parser.add_argument("--n-train", type=int, default=100_000)
    parser.add_argument("--n-test", type=int, default=40_000)
    parser.add_argument("--n-seeds", type=int, default=2)
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--tfidf-mode", default="word", choices=("word", "char", "both"))
    parser.add_argument("--tfidf-components", type=int, default=96)
    parser.add_argument("--tfidf-min-df", type=int, default=5)
    parser.add_argument("--use-phobert", action="store_true")
    parser.add_argument("--out-dir", default="outputs/text_features")
    args = parser.parse_args(argv)

    config = BenchmarkConfig(
        n_train=args.n_train,
        n_test=args.n_test,
        seed=args.seed,
        n_seeds=args.n_seeds,
        tfidf_mode=args.tfidf_mode,
        tfidf_components=args.tfidf_components,
        tfidf_min_df=args.tfidf_min_df,
        use_phobert=args.use_phobert,
        phobert={"n_components": 64, "max_length": 128, "batch_size": 16} if args.use_phobert else {},
    )
    report = run_uplift_benchmark(
        args.train_shard, args.test_shard, config=config, cache_dir=args.cache_dir
    )
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "uplift_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
    (out_dir / "uplift_report.md").write_text(format_report(report) + "\n")
    print(format_report(report))
    print(f"\nwrote {out_dir/'uplift_report.json'} and {out_dir/'uplift_report.md'}")
    return 0

In [ ]:
import numpy as np
import pandas as pd

# ENTITY_COLUMNS, TARGET_LEAKING_ENTITIES, KeywordExtractor, TextFeatureConfig,
# TextFeaturePipeline, TfidfTextEncoder, clean_frame, extract_entities,
# load_shard_sample, parse_vn_number, prepare_for_tfidf, time_span, resolve_shard,
# NUMERIC_COLUMNS, impute_from_text, BenchmarkConfig, entity_agreement_report,
# format_report, run_uplift_benchmark, TabularEncoder
# are all defined by the "Bundled library" cells above.

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

## 1. Load a sample and check the data before modelling

`resolve_shard` downloads on first use and caches under `DATA_DIR`.

In [ ]:
DATA_DIR = "data"
TRAIN_SHARD, TEST_SHARD = "shard_0000", "shard_0009"

raw_train = load_shard_sample(TRAIN_SHARD, n_rows=60_000, seed=0, cache_dir=DATA_DIR)
raw_test = load_shard_sample(TEST_SHARD, n_rows=25_000, seed=1, cache_dir=DATA_DIR)
print("train", len(raw_train), time_span(raw_train))
print("test ", len(raw_test), time_span(raw_test))
raw_train.head(3)

### Three things the raw data does not tell you

Each of these was verified on the published shards and each one changes how you should model the target.

In [ ]:
import pyarrow.parquet as pq

schema = pq.ParquetFile(resolve_shard(TRAIN_SHARD, cache_dir=DATA_DIR)).schema_arrow
print("1) `price` is declared float64 on the dataset card but stored as", schema.field("price").type)
print("   parse_price() converts it; non-numeric values become NaN, never a wrong number.\n")

flags = KeywordExtractor().extract(raw_train)
mentions = flags["kw_cho_thue"]
p = raw_train["price"]
print(f"2) {mentions.mean():.1%} of rows MENTION 'cho thuê' -- that does not make them rentals:")
print(f"   median price, mentioning rows : {p[mentions].median():,.0f} VND")
print(f"   median price, other rows      : {p[~mentions].median():,.0f} VND")
print(f"   share of mentioning rows at sale scale (>= 500M VND): {(p[mentions] >= 5e8).mean():.1%}")
print("   -> kw_cho_thue is a FEATURE, not a filter. Real rentals quote a monthly rate")
print("      (15-60M VND) and are excluded by clean_frame's price floor instead.\n")

clean = clean_frame(raw_train)
print(f"3) clean_frame applies the EDA's Rule 1 (price 100M-200B, area 15-3000 m2,")
print(f"   unit price 3M-800M/m2): kept {len(clean):,}/{len(raw_train):,} = {len(clean)/len(raw_train):.2%}\n")

sparse = raw_train[["floor_count", "frontage_width", "house_depth", "road_width",
                    "bedroom_count", "bathroom_count"]].isna().mean().sort_values(ascending=False)
print("4) structured columns are heavily sparse -- but the text usually states these values:")
print(sparse.map(lambda v: f"{v:.1%}").to_string())
print(f"\nraw price range on this shard: {p.min():,.0f} .. {p.max():,.0f} VND")

## 2. Domain keyword flags

Patterns run against diacritic-folded text, so `"Ô Tô Đỗ Cửa"`, `"ô tô đỗ cửa"`, `"ôtô đỗ cổng"` and `"o to do cua"` all match the same rule.

The prevalence table below doubles as a lexicon health check: a rule that never fires on real data is dead weight, and the test suite fails if any pattern contains a literal diacritic (which could never match folded text).

In [ ]:
extractor = KeywordExtractor()
kw_train = extractor.extract(raw_train)
print(f"{kw_train.shape[1]} flags, {int((kw_train.sum(axis=1) == 0).mean() * 100)}% of rows match nothing\n")
extractor.prevalence(kw_train).head(28)

In [ ]:
# Spot-check the phrases named in the requirements.
demo = pd.DataFrame({"description": [
    "Bán nhà mặt tiền, sổ đỏ chính chủ, ô tô đỗ cửa, pháp lý rõ ràng",
    "Nhà trong ngõ ba gác, nội thất cơ bản, đang chờ sổ",
    "Không tranh chấp, không quy hoạch, xe hơi vào nhà, full nội thất",
    "Nhà cấp 4 cũ, hẻm nhỏ",
]})
cols = ["kw_so_do", "kw_chinh_chu", "kw_mat_tien", "kw_o_to_do_cua", "kw_ngo_ba_gac",
        "kw_xe_hoi_vao_nha", "kw_noi_that_co_ban", "kw_dang_cho_so", "kw_nha_cap_4",
        "kw_tranh_chap", "kw_quy_hoach", "kw_khong_quy_hoach"]
KeywordExtractor().extract(demo)[cols].assign(description=demo["description"])

Row 3 is the reason the negation guard exists: `không tranh chấp` / `không quy hoạch` advertise the *absence* of an encumbrance, so `kw_tranh_chap` and `kw_quy_hoach` stay `False` while `kw_khong_quy_hoach` turns on.

Note that `kw_cho_thue` is a **mention** flag, not a listing-type classifier — see §1. Do not use it to filter rows.

## 3. Numeric entities and text-driven imputation

`extract_entities` parses the numbers listings state in prose. Because the structured columns are sparse, most of the value here is **recovery**: filling `house_depth`, `floor_count` and `road_width` where the source record has nothing.

In [ ]:
ent_train = extract_entities(raw_train)
ent_train.describe().T[["count", "mean", "50%"]]

In [ ]:
# Validation: where the structured column and the parsed entity both exist, do they agree?
entity_agreement_report(raw_train, ent_train)

Reading the table: `text_coverage` is how often the text yields a value, `col missing` is how often the structured column is empty, `agreement` is exact-match rate where both exist, and `recovered` is the number of previously-null cells the text fills.

`N lầu` is deliberately kept in its own `text_lau_count` column instead of being added to `floor_count`: southern listings use it both for "levels above ground" and "levels total", so converting it would bake in an unfounded assumption.

In [ ]:
# Vietnamese number parsing: both separators, and 3-digit groups read as thousands.
for token in ["4", "7.45", "1,3", "1.300", "1.300.000"]:
    print(f"  {token:>12}  ->  {parse_vn_number(token):,.2f}")

## 4. Why `text_price_vnd` is measured but never modelled

The advertised price is written into the listing itself (`"giá: 7.45 tỷ"`), so parsing it back out and feeding it to a model trained on `price` is circular — it scores transcription, not valuation. In a first run this single column took **57% of total split gain** and flattered the whole entities arm by ~26% RMSLE.

Two controls are in place:

- `TARGET_LEAKING_ENTITIES` is excluded from every model feature block.
- Price mentions are **redacted from the TF-IDF/PhoBERT input** too, otherwise digit n-grams smuggle the same information back in.

The cell below is the direct check: two identical listings quoting different prices must encode *identically* once redaction is on.

In [ ]:
BASE = "Nhà mặt tiền sổ đỏ chính chủ, ô tô đỗ cửa, diện tích 40m2, 3 phòng ngủ"
print("redacted :", prepare_for_tfidf(f"{BASE}, giá 5.5 tỷ"))
print("unredacted:", prepare_for_tfidf(f"{BASE}, giá 5.5 tỷ", redact_price=False))

demo_frame = pd.DataFrame({"name": ["A", "B"], "description": [
    f"{BASE}, giá 5.5 tỷ", f"{BASE}, giá 7.25 tỷ"]})
for redact in (True, False):
    cfg = TextFeatureConfig(use_keywords=False, use_entities=False,
                            tfidf_components=2, tfidf_min_df=1, redact_price=redact)
    vec = TextFeaturePipeline(cfg).fit_transform(demo_frame).to_numpy(dtype=float)
    identical = np.allclose(vec[0], vec[1], atol=1e-6)
    print(f"redact_price={redact!s:5} -> price-only difference visible to TF-IDF: {not identical}")

## 5. TF-IDF representation

Vietnamese is syllable-delimited, so word **bigrams** are what capture domain phrases like `sổ đỏ` and `mặt tiền`. The character view is a robustness alternative; §7 compares them.

The encoder is fitted on training text only and reused unchanged on the test period.

In [ ]:
corpus = [prepare_for_tfidf(t) for t in
          (raw_train["name"].fillna("") + " " + raw_train["description"].fillna(""))]

tfidf = TfidfTextEncoder(mode="word", n_components=32, min_df=10).fit(corpus[:20_000])
print("vocabulary:", tfidf.vocabulary_sizes)
print("explained variance:", {k: round(v, 3) for k, v in tfidf.explained_variance.items()})

for component in (0, 1):
    terms = ", ".join(t for t, _ in tfidf.top_terms("w", component, k=10))
    print(f"\ncomponent {component}: {terms}")

## 6. Marginal uplift benchmark

One reference tabular block (numerics, ratios, frequency + smoothed target encodings of geography) trained with LightGBM on `log1p(price)`, then the same block plus each text family in turn. Every arm is a superset of the previous one, so each row of the delta table is a **marginal** gain.

Early stopping uses the latest 15% of the *training* period, never the test period. Two seeds per arm, because an uplift smaller than the seed-to-seed spread is not a finding.

The tabular block here is a reference harness, not the project baseline — `run_uplift_benchmark` accepts a pre-built matrix so the HYPE-12 pipeline can be dropped in.

In [ ]:
config = BenchmarkConfig(
    n_train=60_000, n_test=25_000, n_seeds=2,
    tfidf_mode="word", tfidf_components=128, tfidf_min_df=5,
    exclude_target_leaking_entities=True, redact_price_in_text=True,
)
report = run_uplift_benchmark(TRAIN_SHARD, TEST_SHARD, config=config, cache_dir=DATA_DIR)
print(format_report(report))

## 7. Comparing TF-IDF analyzers

Word bigrams against character 3–5-grams against both concatenated, at matched sample sizes.

In [ ]:
rows = []
for mode in ("word", "char", "both"):
    cfg = BenchmarkConfig(n_train=40_000, n_test=20_000, n_seeds=1,
                          tfidf_mode=mode, tfidf_components=128)
    rep = run_uplift_benchmark(TRAIN_SHARD, TEST_SHARD, config=cfg, cache_dir=DATA_DIR)
    for arm in ("tabular", "+tfidf"):
        rows.append({"mode": mode, "arm": arm, **rep["arms"][arm]["metrics"]})

comp = pd.DataFrame(rows).drop(columns=["n"])
comp

In [ ]:
gain = (comp[comp.arm == "+tfidf"].set_index("mode")["rmsle"]
        - comp[comp.arm == "tabular"].set_index("mode")["rmsle"])
print("RMSLE change from adding TF-IDF, by analyzer (negative is better):")
print(gain.map(lambda v: f"{v:+.4f}").to_string())

## 8. Optional: PhoBERT embeddings

Heavier and opt-in. `torch`/`transformers` are imported lazily, so nothing above needs them. Mean-pooled `vinai/phobert-base`, reduced with SVD, on a subsample — CPU encoding is the bottleneck.

Diacritics are preserved on this path (unlike TF-IDF) because PhoBERT's vocabulary distinguishes them; price mentions are still redacted.

In [ ]:
%pip install -q torch --index-url https://download.pytorch.org/whl/cpu
%pip install -q transformers

cfg = BenchmarkConfig(
    n_train=4_000, n_test=2_000, n_seeds=1,
    tfidf_mode="word", tfidf_components=128, use_phobert=True,
    phobert={"n_components": 64, "max_length": 128, "batch_size": 16},
)
phobert_report = run_uplift_benchmark(TRAIN_SHARD, TEST_SHARD, config=cfg, cache_dir=DATA_DIR)
print(format_report(phobert_report))

## 9. Using these features in the baseline pipeline

Fit the pipeline on training rows only, transform both sides, and column-bind onto the tabular block. The index contract is what keeps this safe: `transform` returns rows aligned to the input frame, and imputation must happen **before** tabular encoding so derived ratios use the recovered numbers.

In [ ]:
# NUMERIC_COLUMNS, impute_from_text and TabularEncoder come from the bundled
# library cells in section 0.1.

# Clean with the EDA's Rule 1 bounds. No rental flag: kw_cho_thue marks a mention,
# and the price floor is what actually excludes monthly-rate rental listings.
train_df = clean_frame(raw_train)
test_df = clean_frame(raw_test)
print("clean rows:", len(train_df), len(test_df))

pipeline = TextFeaturePipeline(TextFeatureConfig(
    tfidf_mode="word", tfidf_components=128, tfidf_min_df=5,
    redact_price=True,          # keep the regression target out of the text features
))
train_features = pipeline.fit_transform(train_df)   # fits TF-IDF/SVD on TRAIN only
test_features = pipeline.transform(test_df)         # reuses that fitted state

# 1. Impute the sparse structured columns from text, THEN encode the tabular
#    block, so derived ratios use the recovered numbers rather than the nulls.
numeric = [c for c in NUMERIC_COLUMNS if c in train_df.columns]
train_imp, test_imp = train_df.copy(), test_df.copy()
train_imp[numeric] = impute_from_text(train_df[numeric], train_features[ENTITY_COLUMNS])[numeric]
test_imp[numeric] = impute_from_text(test_df[numeric], test_features[ENTITY_COLUMNS])[numeric]

encoder = TabularEncoder().fit(train_imp)
tabular_train = encoder.transform(train_imp)
tabular_test = encoder.transform(test_imp)

# 2. Bind the text features on, minus the target-restating entity.
drop = [c for c in TARGET_LEAKING_ENTITIES if c in train_features.columns]
X_train = pd.concat([tabular_train, train_features.drop(columns=drop)], axis=1)
X_test = pd.concat([tabular_test, test_features.drop(columns=drop)], axis=1)

assert list(X_train.columns) == list(X_test.columns)
assert len(X_train) == len(train_df) and len(X_test) == len(test_df)
print("tabular:", tabular_train.shape[1], "-> with text:", X_train.shape[1], "features")

## 10. Takeaways

Measured out-of-time (train 2025-06 → test 2026-03). Full tables in `reports/text_features/` and `docs/real_estate_text_features.md`.

- **Cumulative text gain over the tabular baseline: −0.0400 RMSLE (−10.07%), −2.67 MAPE points, +0.028 R²** on 92,855 train / 36,118 test rows. Ladder: tabular 0.3973 → +keywords 0.3765 → +entities 0.3678 → +tfidf **0.3573**. Every arm improves every metric. Seed spread is ≤ 0.0006 RMSLE, so each step is 30–100× the noise.
- **The biggest win is entity recovery, not bag-of-words.** `house_depth` is 97.0% null in the source and text fills 26,841 cells; `floor_count` is 82.7% null and text fills 26,179. Parsed values agree with the recorded ones in 86–95% of overlapping rows.
- **Keyword flags are cheap and additive** (−5.22% RMSLE) but individually weak: geographic target encodings already absorb much of the same signal.
- **Word bigrams beat character n-grams.** At matched scale (55,713/22,557, same tabular baseline 0.4144): word 0.3694, both 0.3710, char 0.3774. Concatenating the char view does *not* beat word alone despite 60% more features.
- **PhoBERT was measured and lost.** Frozen mean-pooled embeddings make RMSLE, MAPE and R² all *worse* than TF-IDF alone, at ~30× the compute per row. Worth revisiting fine-tuned; do not pay for pooled PhoBERT by default.
- **`cho thuê` in the text is a mention, not a listing type.** It fires on 18.8% of rows, but 96% of those are sales at a sale-scale median price (10 tỷ) pitching rental yield — "sẵn hợp đồng thuê". Use it as a feature; never as a row filter. Real rentals quote a monthly rate and are removed by the price floor.
- **Leakage is the main hazard of text features here.** The asking price is in the prose; unredacted it took 57% of total split gain. Redact it or the benchmark measures transcription, not valuation.
- **Clean with the EDA's Rule 1** (price 100M–200B, area 15–3000 m², unit price 3M–800M/m²). That retains ~93% of rows and lifts raw-scale R² to ~0.75; looser bounds leave an outlier tail past 800 trillion VND that crushes R² to ~0.05 and makes MAE meaningless.